# New version, using the .parquet exports
instead of synchronizing the complete postgresql database all the time, i can export PARAM and METRIC .parquet files per group_id and only synchronize/copy what i need.

In [ ]:
import sys
import os
project_root = os.path.abspath('..')          # one level up from notebooks' directory
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import polars as pl
import dask.dataframe as dd
import numpy as np
from pathlib import Path
from key_mapping import key_mapping
from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics

updating value_mapping...


In [2]:
# tags =["2026-03-31_12-34_hpc3_P3.1.10", "P3.1.14", "P3.1.15", "P3.1.16-v1", "P3.1.17-v2"]
# tags = ["P3.1.17"]
# tags = ["P3.1.16-new"]
# tags = ["P3.1.17-v2"]
tags = ["P3.1.17-v3"]
PARAMS = []
METRICS = []
for tag in tags:
    params = dd.read_parquet(f"../data/experiments/exports/['{tag}']-params.parquet", )
    PARAMS.append(params)
    metrics = dd.read_parquet(f"../data/experiments/exports/['{tag}']-metrics.parquet", )
    METRICS.append(metrics)
PARAMS = dd.concat(PARAMS)
METRICS = dd.concat(METRICS)

In [7]:
METRICS.sample(frac=0.1).compute()

,run_uuid,step,key,value
13967531,8768d09a2b044900b49b535f99a0939a,171,gain_per_reallocation_utilization,NaN
27846006,fed842721dd548f8b0fa214fc9946100,118,mean_squared_error,NaN
89319666,8517bea132f74345bbe6cc7cf5e374b5,242,rel_num_inf_bids,NaN
86875480,05b88683258240b8bc5e91f77c45c8a8,88,rel_num_feas_bids,NaN
58243931,8d3fb3b810c94a14aa0125c623d48c83,91,num_bids,NaN
...,...,...,...,...
108879349,740f023a23974055a4113fe4689e3d09,245,target_opt_2,50.675592
52595664,18fdf245c281424bb11238858445f2c2,208,my_tsp_obj_val_diff_0,NaN
54317740,262358d4fd0343bfac33ce4c0d22f518,172,my_tsp_obj_val_diff_1,NaN
38566747,8b081c7fb29740c8a4eb661e536b7516,91,my_hausdorff_distance_0,NaN


In [ ]:
single_step_metrics = (
    METRICS.filter(
        pl.col('value').is_not_null())
        .groupby('key')
        .agg()

run_uuid,step,key,value
str,i64,str,f64
"""00067e60671c4831809ad06154378e…",0,"""degree_of_reallocation""",0.0
"""000ba6aa724747cdade16870bcd73a…",0,"""degree_of_reallocation""",0.625
"""0019e9cc468441658e7caddcdbf1cb…",0,"""degree_of_reallocation""",0.625
"""00227e75d2284afb8742144755c15a…",0,"""degree_of_reallocation""",0.0
"""004478b225fe4d978ff66f0838533e…",0,"""degree_of_reallocation""",0.0
…,…,…,…
"""ffcf3736663848e09986adc2ee0b6a…",0,"""target_opt_final_2""",47.386707
"""ffd75aac2e44428bbd6f95e926585c…",0,"""target_opt_final_2""",10.148892
"""ffeb1a2c10374e9f97b54bcec03f1b…",0,"""target_opt_final_2""",16.660207


In [ ]:
METRICS.filter(pl.col('key') == 'degree_of_reallocation')

run_uuid,step,key,value
str,i64,str,f64
"""00067e60671c4831809ad06154378e…",0,"""degree_of_reallocation""",0.0
"""00067e60671c4831809ad06154378e…",1,"""degree_of_reallocation""",null
"""00067e60671c4831809ad06154378e…",2,"""degree_of_reallocation""",null
"""00067e60671c4831809ad06154378e…",3,"""degree_of_reallocation""",null
"""00067e60671c4831809ad06154378e…",4,"""degree_of_reallocation""",null
…,…,…,…
"""fff74fe75baf48b78396af4d5f7243…",251,"""degree_of_reallocation""",null
"""fff74fe75baf48b78396af4d5f7243…",252,"""degree_of_reallocation""",null
"""fff74fe75baf48b78396af4d5f7243…",253,"""degree_of_reallocation""",null


In [ ]:
# # polars version
# q1 = (
#     pl.scan_parquet()
#     .pivot()
#     .filter(pl.col('step')>0)
#     .
# )
# METRICS_WIDE = (
#     METRICS.pivot(on='key', values='value', index=['run_uuid', 'step'])
# )
# step_gt0 = METRICS_WIDE.filter(pl.col('step') > 0)
# single_step_metrics = [
#     c for c in METRICS_WIDE.columns
#     if step_gt0.select(pl.col(c).is_null().all()).item()
# ]


In [ ]:
# pandas version
from key_mapping import key_mapping
from data_preparation import add_grouped_mean_columns, get_multi_carrier_metrics

METRICS_WIDE = METRICS.pivot(columns='key', index=['run_uuid', 'step'], values='value').reset_index()
single_step_metrics = [col for col in METRICS_WIDE.columns if pd.isna(METRICS_WIDE[METRICS_WIDE['step'] > 0][col]).all()]
SS_METRICS = METRICS_WIDE[METRICS_WIDE['step'] == 0][['run_uuid'] + single_step_metrics]
SS_MERGED = pd.merge(PARAMS, SS_METRICS, on='run_uuid')
add_grouped_mean_columns(SS_MERGED, inplace=True)

# single step, multi-carrier metrics
SS_MC_METRICS = get_multi_carrier_metrics(SS_METRICS)
SS_MC_MERGED = pd.merge(PARAMS, SS_MC_METRICS, on='run_uuid')

# Multi-step metrics
multi_step_metrics = [col for col in METRICS_WIDE.columns if col not in single_step_metrics]
MS_METRICS = METRICS_WIDE[multi_step_metrics]
MS_METRICS.update(MS_METRICS.groupby('run_uuid').ffill())  # forward fill the multi-step metrics that might have gaps
MS_METRICS = MS_METRICS.replace(np.inf, np.nan)
add_grouped_mean_columns(MS_METRICS, inplace=True)
MS_MERGED = pd.merge(PARAMS, MS_METRICS, on='run_uuid')

# to wide format
SS_MC_MERGED_WIDE = SS_MC_MERGED.pivot(columns='key', values='value', index=PARAMS.columns.to_list() + ['carrier']).reset_index()
SS_MC_MERGED_WIDE = SS_MC_MERGED_WIDE.rename(columns=key_mapping)

In [ ]:
non_plot_params = [
    'run_uuid',
    'name_x',
    'source_type',
    'source_name',
    'entry_point_name',
    'user_id',
    'status',
    'start_time',
    'end_time',
    'source_version',
    'lifecycle_stage_x',
    'artifact_uri',
    'experiment_id',
    'deleted_time',
    'name_y',
    'artifact_location',
    'lifecycle_stage_y',
    'creation_time',
    'last_update_time',
    'group_id',
    'instance_id',
    'mlflow.runName',
    'mlflow.source.git.commit',
    'mlflow.source.name',
    'mlflow.source.type',
    'mlflow.user',
    'mlflow.loggedArtifacts',
]
plot_params = [x for x in PARAMS.columns if x not in non_plot_params ]
non_unique_params: pd.DataFrame = PARAMS.loc[:, PARAMS.nunique() != 1]
non_unique_params = non_unique_params.drop(columns=non_plot_params, errors='ignore', inplace=False)
print("List of non-unique parameters:")
list(non_unique_params.columns)
# sort non-unique first
plot_params = [None] + list(non_unique_params) + [x for x in plot_params if x not in non_unique_params]

# get all metrics
plot_metrics = (SS_MC_MERGED["key"].unique())

## Plots

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
import plotnine as p9
from my_theming import my_p9_theme, my_scale_color_and_fill

# 2. Define the plotting function
def update_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10):
    ylab=df["Error Function"][0] if y == "target_opt_final" else y
    if fill in [None, "None"]:
        if linetype in [None, "None"]:
            my_aes = p9.aes(x, y)
        else:
            my_aes = p9.aes(x, y, linetype=linetype)
    elif linetype in [None, "None"]:
        my_aes=p9.aes(x, y, fill=fill)
    else:
        my_aes=p9.aes(x, y, fill=fill, linetype=linetype)

    # Start building the plot
    p = (
        p9.ggplot(df,
                    my_aes)
                #   p9.aes(x=x, y=y, fill=fill, linetype=linetype))
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(12, 6))
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )
    if scale_y_log10:
        p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(
        axis_text_x=p9.element_blank(),
        axis_ticks_x=p9.element_blank(),
        axis_title_x=p9.element_blank(),)
    
    # 3. Add faceting logic only if a parameter is selected
    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    group_keys = [key for key in [fill, cols, rows, linetype] if key not in [None, "None"]]
    n_per_series = df.groupby(group_keys)[y].count()
    print(f"n per boxplot:\n{n_per_series}")

    return p.draw() # Using .draw() ensures it renders correctly in the widget output

# 4. Create the interactive interface
interact(
    update_plot,
    df=widgets.fixed(SS_MC_MERGED_WIDE),
    x=widgets.Dropdown(options=plot_params, value=None, description='X Axis:'),
    y=widgets.Dropdown(options=plot_metrics, value="target_opt_final", description='Y Axis:'),
    fill=widgets.Dropdown(options=plot_params, value="Optimizer", description='Fill Color:'),
    linetype=widgets.Dropdown(options=plot_params, value="Optimizer", description='Linetype'),
    cols=widgets.Dropdown(options=plot_params, value="# clusters", description='Facet Cols:'),
    rows=widgets.Dropdown(options=plot_params, value="data__num_requests_per_carrier", description='Facet Rows:'),
    hide_x_axis=widgets.Checkbox(value=False, description="Hide x axis"),
    scale_y_log10=widgets.Checkbox(value=False, description="Log y axis"),
);

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotnine as p9
import os

# --- 1. Global holder for the current plot object ---
current_p = [None]

# --- 2. Define the plotting function logic ---
def create_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10):
    ylab = df["Error Function"][0] if y == "target_opt_final" else y
    
    # Aesthetic logic
    aes_kwargs = {'x': x, 'y': y}
    if fill not in [None, "None"]:
        aes_kwargs['fill'] = fill
    if linetype not in [None, "None"]:
        aes_kwargs['linetype'] = linetype
    
    my_aes = p9.aes(**aes_kwargs)

    p = (
        p9.ggplot(df, my_aes)
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(12, 6))
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )
    
    if scale_y_log10:
        p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(
            axis_text_x=p9.element_blank(),
            axis_ticks_x=p9.element_blank(),
            axis_title_x=p9.element_blank(),
        )
    
    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    # Store for saving later
    current_p[0] = p
    
    # Print stats
    group_keys = [key for key in [fill, cols, rows, linetype] if key not in [None, "None"]]
    if group_keys:
        n_per_series = df.groupby(group_keys)[y].count()
        print(f"n per boxplot:\n{n_per_series}")
    
    return p

# --- 3. Define Widgets ---
ui_x = widgets.Dropdown(options=plot_params, value=None, description='X Axis:')
ui_y = widgets.Dropdown(options=plot_metrics, value="target_opt_final", description='Y Axis:')
ui_fill = widgets.Dropdown(options=plot_params, value="Optimizer", description='Fill Color:')
ui_line = widgets.Dropdown(options=plot_params, value="Optimizer", description='Linetype:')
ui_cols = widgets.Dropdown(options=[None] + plot_params, value="# clusters", description='Cols:')
ui_rows = widgets.Dropdown(options=[None] + plot_params, value="data__num_requests_per_carrier", description='Rows:')
ui_hide_x = widgets.Checkbox(value=False, description="Hide x axis")
ui_log = widgets.Checkbox(value=False, description="Log y axis")

# Save Widgets
ui_filename = widgets.Text(value='my_plot.png', description='Filename:')
ui_save_btn = widgets.Button(description="💾 Save Figure", button_style='success')
ui_save_output = widgets.Output()

# --- 4. Event Handlers ---
def on_save_clicked(b):
    with ui_save_output:
        clear_output()
        if current_p[0] is not None:
            fname = ui_filename.value
            current_p[0].save(fname, verbose=False)
            print(f"✅ Saved to {os.path.abspath(fname)}")
        else:
            print("❌ No plot generated yet!")

ui_save_btn.on_click(on_save_clicked)

# --- 5. Assemble the Dashboard ---
out = widgets.interactive_output(
    create_plot, 
    {
        'df': widgets.fixed(SS_MC_MERGED_WIDE),
        'x': ui_x, 'y': ui_y, 'fill': ui_fill, 'linetype': ui_line,
        'cols': ui_cols, 'rows': ui_rows, 
        'hide_x_axis': ui_hide_x, 'scale_y_log10': ui_log
    }
)

# Layout: UI controls on top/left, Save bar at bottom
controls = widgets.VBox([
    widgets.HBox([ui_x, ui_y, ui_fill]),
    widgets.HBox([ui_line, ui_cols, ui_rows]),
    widgets.HBox([ui_hide_x, ui_log]),
    widgets.HBox([ui_filename, ui_save_btn, ui_save_output])
])

display(controls, out)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotnine as p9
import os

# --- 1. Global holder for the current plot object ---
current_p = [None]

# --- 2. Define the plotting function logic ---
def create_plot(df, x, y, fill, linetype, cols, rows, hide_x_axis, scale_y_log10):
    # (Setting up labels and aesthetics - same as your logic)
    ylab = df["Error Function"][0] if y == "target_opt_final" else y
    
    aes_kwargs = {'x': x, 'y': y}
    if fill not in [None, "None"]: aes_kwargs['fill'] = fill
    if linetype not in [None, "None"]: aes_kwargs['linetype'] = linetype
    
    my_aes = p9.aes(**aes_kwargs)

    p = (
        p9.ggplot(df, my_aes)
        + p9.geom_boxplot()
        + my_p9_theme
        + p9.theme(figure_size=(12, 6))
        + my_scale_color_and_fill
        + p9.labs(y=ylab)
    )
    
    if scale_y_log10: p += p9.scale_y_log10()
    if hide_x_axis:
        p += p9.theme(axis_text_x=p9.element_blank(), 
                      axis_ticks_x=p9.element_blank(), 
                      axis_title_x=p9.element_blank())
    
    if cols or rows:
        p += p9.facet_grid(rows=rows, cols=cols)

    # Update the global reference for the save button
    current_p[0] = p
    
    # --- CRITICAL CHANGE START ---
    # In interactive_output, we must explicitly display the plot
    display(p) 
    
    # Print stats below the plot
    group_keys = [key for key in [fill, cols, rows, linetype] if key not in [None, "None"]]
    if group_keys:
        n_per_series = df.groupby(group_keys)[y].count()
        print(f"\nn per boxplot:\n{n_per_series}")
    # --- CRITICAL CHANGE END ---

# --- 3. Define Widgets (Keep these exactly as they were) ---
ui_x = widgets.Dropdown(options=plot_params, value=None, description='X Axis:')
ui_y = widgets.Dropdown(options=plot_metrics, value="target_opt_final", description='Y Axis:')
ui_fill = widgets.Dropdown(options=plot_params, value="Optimizer", description='Fill Color:')
ui_line = widgets.Dropdown(options=plot_params, value="Optimizer", description='Linetype:')
ui_cols = widgets.Dropdown(options=[None] + plot_params, value="# clusters", description='Facet Cols:')
ui_rows = widgets.Dropdown(options=[None] + plot_params, value="data__num_requests_per_carrier", description='Facet Rows:')
ui_hide_x = widgets.Checkbox(value=False, description="Hide x axis")
ui_log = widgets.Checkbox(value=False, description="Log y axis")

ui_filename = widgets.Text(value='my_plot.png', description='Filename:')
ui_save_btn = widgets.Button(description="💾 Save Figure", button_style='success')
ui_save_output = widgets.Output()

def on_save_clicked(b):
    with ui_save_output:
        clear_output()
        if current_p[0] is not None:
            current_p[0].save(ui_filename.value, verbose=False)
            print(f"✅ Saved!")
        else:
            print("❌ Error")

ui_save_btn.on_click(on_save_clicked)

# --- 4. Linking and Displaying ---
# This widget captures the output of 'create_plot'
out = widgets.interactive_output(
    create_plot, 
    {
        'df': widgets.fixed(SS_MC_MERGED_WIDE),
        'x': ui_x, 'y': ui_y, 'fill': ui_fill, 'linetype': ui_line,
        'cols': ui_cols, 'rows': ui_rows, 
        'hide_x_axis': ui_hide_x, 'scale_y_log10': ui_log
    }
)

controls = widgets.VBox([
    widgets.HBox([ui_x, ui_y, ui_fill]),
    widgets.HBox([ui_line, ui_cols, ui_rows]),
    widgets.HBox([ui_hide_x, ui_log]),
    widgets.HBox([ui_filename, ui_save_btn, ui_save_output])
])

# Use display() to show the UI and the output area
display(controls, out)